In [1]:
!pip install langchain langchain-community langchain-text-splitters langchain-chroma chromadb sentence-transformers pypdf pydantic langgraph

   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.4 MB ? eta -:--:--
   -------- ------------------------------- 0.5/2.4 MB 1.6 MB/s eta 0:00:02
   -------- ------------------------------- 0.5/2.4 MB 1.6 MB/s eta 0:00:02
   -------- ------------------------------- 0.5/2.4 MB 1.6 MB/s eta 0:00:02
   -------- ------------------------------- 0.5/2.4 MB 1.6 MB/s eta 0:00:02
   ------------- -------------------------- 0.8/2.4 MB 564.6 kB/s eta 0:00:03
   ------------- -------------------------- 0.8/2.4 MB 564.6 kB/s eta 0:00:03
   ----------------- ---------------------- 1.0/2.4 MB 572.9 kB/s eta 0:00:03
   ---------------------- ----------------- 1.3/2.4 MB 615.9 kB/s eta 0:00:02
   -------------------------- ------------- 1.6/2.4 MB 681.3 kB/s eta 0:00:02
   -------------------------- ------------- 1.6/2.4 MB 681.3 kB/s eta 0:00:02
   ------------------

In [2]:
import os
import json
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

from pydantic import BaseModel, Field
from typing import Optional, List, Dict

C:\Users\krish\AppData\Local\Temp\ipykernel_8236\3883374866.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
BASE_DIR = Path.cwd()

KNOWLEDGE_DIR = BASE_DIR / "knowledge_base"
DATA_DIR = BASE_DIR / "data"
APP_DIR = BASE_DIR / "app"

KNOWLEDGE_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)
APP_DIR.mkdir(exist_ok=True)

print("Project folders ready.")
print("Knowledge base:", KNOWLEDGE_DIR)

Project folders ready.
Knowledge base: C:\Users\krish\Downloads\Biodiversity Chatbot\knowledge_base


In [4]:
from langchain_community.document_loaders import PyPDFLoader

pdf_files = list(KNOWLEDGE_DIR.glob("*.pdf"))

print(f"Found {len(pdf_files)} PDF files:")

for pdf in pdf_files:
    print("-", pdf.name)

Found 4 PDF files:
- agroforestry_primer.pdf
- fao_soil_carbon.pdf
- ipbes_biodiversity.pdf
- ipcc_technical_summary.pdf


In [7]:
documents = []

for pdf_path in pdf_files:
    print(f"Loading: {pdf_path.name}")

    loader = PyPDFLoader(str(pdf_path))
    pages = loader.load()


    for page in pages:
        page.metadata["source_file"] = pdf_path.name

    documents.extend(pages)

print("\nTotal pages loaded:", len(documents))

Loading: agroforestry_primer.pdf
Loading: fao_soil_carbon.pdf
Loading: ipbes_biodiversity.pdf
Loading: ipcc_technical_summary.pdf

Total pages loaded: 371


In [8]:
print(documents[0].page_content[:1000])

iAgroforestry: a primer
AGROFORESTRY:
A PRIMER
Design and management principles for 
people and the environment
Editors
Anja Gassner and Philip Dobie


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Total chunks created:", len(chunks))

Total chunks created: 1108


In [10]:
print(chunks[0].page_content)
print("\nMetadata:")
print(chunks[0].metadata)

iAgroforestry: a primer
AGROFORESTRY:
A PRIMER
Design and management principles for 
people and the environment
Editors
Anja Gassner and Philip Dobie

Metadata:
{'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.0 (Macintosh)', 'creationdate': '2022-12-16T10:58:08+07:00', 'moddate': '2022-12-16T11:00:04+07:00', 'trapped': '/False', 'source': 'C:\\Users\\krish\\Downloads\\Biodiversity Chatbot\\knowledge_base\\agroforestry_primer.pdf', 'total_pages': 181, 'page': 0, 'page_label': 'i', 'source_file': 'agroforestry_primer.pdf'}


In [11]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model ready!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model ready!


In [12]:
from langchain_chroma import Chroma

CHROMA_DIR = str(BASE_DIR / "chroma_db")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_DIR,
    collection_name="biodiversity_knowledge"
)

print("Vector database created!")
print("Saved at:", CHROMA_DIR)

Vector database created!
Saved at: C:\Users\krish\Downloads\Biodiversity Chatbot\chroma_db


In [13]:
query = "How does low soil organic carbon affect soil health and biodiversity?"

results = vectorstore.similarity_search(query, k=4)

print("Retrieved:", len(results), "chunks\n")

for i, doc in enumerate(results, 1):
    print("=" * 70)
    print(f"RESULT {i}")
    print("Source:", doc.metadata.get("source_file"))
    print("Page:", doc.metadata.get("page"))
    print()
    print(doc.page_content[:700])
    print()

Retrieved: 4 chunks

RESULT 1
Source: fao_soil_carbon.pdf
Page: 7

VI
SOIL ORGANIC CARBON  the hidden potential
EXECUTIVE SUMMARY
In the presence of climate change, land degradation and biodiversity loss, soils have 
become one of the most vulnerable resources in the world. Soils are a major carbon 
reservoir containing more carbon than the atmosphere and terrestrial vegetation 
combined. Soil organic carbon (SOC) is dynamic, however, and anthropogenic 
impacts on soil can turn it into either a net sink or a net source of GHGs. Enormous 
scientific progress has been achieved in understanding and explaining SOC dynamics. 
Yet, protection and monitoring of SOC stocks at national and global levels still face 
complicated challenges impeding effective on-the-gr

RESULT 2
Source: fao_soil_carbon.pdf
Page: 23

12
SOIL ORGANIC CARBON  the hidden potential
LAND-USE MANAGEMENT 
EXTERNAL DRIVERS
CLIMATE CHANGE,
NITROGEN
DEPOSITION,
INVASIVE SPECIES,
AND POLLUTION
HUMAN
HEALTH
ANIMAL
HEALTH
PLANT

In [14]:
!pip install -q langchain-groq

In [15]:
import os
import getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

Enter your Groq API key:  ········


In [20]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

print("LLM ready!")

LLM ready!


In [21]:
def retrieve_evidence(query, k=6):
    return vectorstore.similarity_search(query, k=k)


def format_evidence(docs):
    formatted = []

    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("source_file", "Unknown")
        page = doc.metadata.get("page", "Unknown")

        formatted.append(
            f"""
SOURCE {i}
File: {source}
Page: {page}

{doc.page_content}
"""
        )

    return "\n".join(formatted)

In [18]:
def scientific_rag_answer(question):
    docs = retrieve_evidence(question, k=6)
    evidence = format_evidence(docs)

    prompt = f"""
You are an environmental science assistant.

Answer the user's question ONLY using the scientific evidence provided below.

Rules:
1. Do not invent scientific facts.
2. Explain relationships between environmental variables.
3. Mention which sources support the answer.
4. If the evidence is insufficient, clearly say so.
5. Keep the explanation understandable to a non-expert.

USER QUESTION:
{question}

SCIENTIFIC EVIDENCE:
{evidence}

Provide:

Scientific Assessment:
...

Environmental Relationships:
...

Evidence:
- source filename, page
"""

    response = llm.invoke(prompt)

    return response.content, docs

In [22]:
answer, retrieved_docs = scientific_rag_answer(
    "How can low soil organic carbon affect biodiversity and soil health?"
)

print(answer)

**Scientific Assessment**  
Low levels of soil organic carbon (SOC) weaken the foundation of healthy soils. SOC is the main food source for the myriad microorganisms, fungi, insects and other organisms that live in the ground. When SOC declines, these soil‑dwelling organisms lose energy and habitat, leading to a drop in their abundance and diversity. Because soil biodiversity drives key processes such as decomposition of plant litter, formation of stable soil aggregates, nutrient cycling, and suppression of pests and diseases, a reduction in SOC ultimately compromises overall soil health, fertility, water‑holding capacity and the ability of the ecosystem to support plant growth.

**Environmental Relationships**

| Low SOC → | Consequence | How it links to biodiversity & soil health |
|----------|-------------|--------------------------------------------|
| **Less food and habitat for microbes** | Decline in microbial abundance and diversity | Soil microbes are the primary agents that c

In [55]:
from pydantic import BaseModel, Field
from typing import Optional

class EnvironmentalProfile(BaseModel):
    soil_organic_carbon: Optional[float] = None
    soil_ph: Optional[float] = None
    soil_moisture: Optional[str] = None

    rainfall: Optional[str] = None
    temperature: Optional[str] = None

    land_use: Optional[str] = None
    crop: Optional[str] = None

    biodiversity_trend: Optional[str] = None
    species_richness: Optional[str] = None
    habitat_diversity: Optional[str] = None

    pollution: Optional[str] = None
    deforestation: Optional[str] = None

    region: Optional[str] = None

In [56]:
profile = EnvironmentalProfile(
    soil_organic_carbon=0.3,
    rainfall="low",
    land_use="agriculture",
    crop="monoculture wheat",
    region="semi-arid"
)

print(profile.model_dump())

{'soil_organic_carbon': 0.3, 'soil_ph': None, 'soil_moisture': None, 'rainfall': 'low', 'temperature': None, 'land_use': 'agriculture', 'crop': 'monoculture wheat', 'biodiversity_trend': None, 'species_richness': None, 'habitat_diversity': None, 'pollution': None, 'deforestation': None, 'region': 'semi-arid'}


In [57]:
def analyze_environmental_profile(profile):
    
    profile_text = json.dumps(
        profile.model_dump(exclude_none=True),
        indent=2
    )

    query = f"""
    Environmental conditions:
    {profile_text}

    Find scientific evidence about the interactions between these
    environmental variables and biodiversity.
    """

    docs = retrieve_evidence(query, k=8)
    evidence = format_evidence(docs)

    prompt = f"""
You are an environmental scientist.

Analyze the environmental profile below using the scientific evidence provided.

ENVIRONMENTAL PROFILE:
{profile_text}

SCIENTIFIC EVIDENCE:
{evidence}

Your task is to reason across MULTIPLE variables together.

Do not analyze each variable independently.

Identify:
1. Main environmental risks
2. Important interactions between variables
3. How these interactions may affect biodiversity
4. Which factors are likely reinforcing each other

Use only claims supported by the evidence.
If evidence is insufficient, say so.

Return:

ENVIRONMENTAL DIAGNOSIS:
...

MULTI-METRIC INTERACTIONS:
1. ...
2. ...
3. ...

BIODIVERSITY IMPACT:
...

EVIDENCE USED:
- source and page
"""

    response = llm.invoke(prompt)

    return response.content, docs

In [58]:
analysis, analysis_docs = analyze_environmental_profile(profile)

print(analysis)

**ENVIRONMENTAL DIAGNOSIS**  
The semi‑arid wheat‑monoculture system is characterized by a very low soil organic carbon (SOC) stock (0.3 % / t ha) together with scarce rainfall.  In this context the principal environmental risks are:

1. **Accelerated soil degradation** – the low SOC value signals a depleted organic matter pool and a reduced capacity of the soil to retain water, nutrients and to support a healthy microbial community.  
2. **Reduced climate‑regulation service** – with little SOC the land’s ability to sequester atmospheric CO₂ is minimal, weakening the “regulation of climate” contribution of ecosystems.  
3. **Loss of habitat and functional biodiversity** – the conversion to a single‑crop, low‑diversity system eliminates native vegetation and associated fauna, shrinking the extent of natural habitat within the agricultural matrix.  

These risks are amplified by the interaction of the four profile variables (low SOC, low rainfall, monoculture wheat, semi‑arid climate) an

In [59]:
def generate_recommendations(profile, diagnosis):

    profile_text = json.dumps(
        profile.model_dump(exclude_none=True),
        indent=2
    )

    search_query = f"""
    Find scientific evidence for interventions that can improve biodiversity,
    soil health, water retention, habitat diversity and resilience under these conditions:

    {profile_text}

    Prefer interventions such as agroforestry, cover crops, intercropping,
    habitat restoration, soil carbon improvement, reduced disturbance,
    or other evidence-supported approaches.
    """

    docs = retrieve_evidence(search_query, k=10)
    evidence = format_evidence(docs)

    prompt = f"""
You are an environmental scientist designing practical biodiversity interventions.

ENVIRONMENTAL PROFILE:
{profile_text}

ENVIRONMENTAL DIAGNOSIS:
{diagnosis}

SCIENTIFIC EVIDENCE:
{evidence}

Generate 3 high-quality recommendations.

IMPORTANT RULES:
- Recommendations must consider multiple environmental variables together.
- Do not give generic advice.
- Use only claims supported by the evidence.
- Do not invent numerical improvement percentages.
- If exact quantitative improvement is unavailable, describe the expected direction of change.
- Clearly state uncertainty.

For EACH recommendation return:

RECOMMENDATION:
What should be done.

WHY IT WORKS:
Scientific reasoning connecting at least 2-3 environmental variables.

IMPACTED METRICS:
- metric: expected direction
- metric: expected direction

TIME HORIZON:
Short term (0-1 year), Medium term (1-3 years), or Long term (3+ years)

CONFIDENCE:
High / Medium / Low

EVIDENCE:
- source filename and page

Also provide a final section:

PRIORITY ORDER:
1. ...
2. ...
3. ...

Explain briefly why this order is recommended.
"""

    response = llm.invoke(prompt)

    return response.content, docs

In [60]:
recommendations, recommendation_docs = generate_recommendations(
    profile,
    analysis
)

print(recommendations)

**RECOMMENDATION 1 – Integrate agro‑forestry strips (nitrogen‑fixing trees, drought‑tolerant shrubs) into the wheat field matrix**  

**WHY IT WORKS**  
- **Plant‑diversity + low rainfall** – Adding deep‑rooted woody species supplies additional organic inputs (leaf litter, fine roots) that increase soil organic carbon (SOC) and improve water‑holding capacity, breaking the low‑SOC ↔ poor‑water‑retention feedback (Interaction 1).  
- **Monoculture + soil‑biodiversity loss** – Woody strips provide heterogeneous root exudates and habitat for soil microbes and fauna, counter‑acting the microbial impoverishment caused by a single‑crop system (Interaction 2).  
- **Land‑use + habitat loss** – The woody strips restore a portion of the “extent of natural habitat in agricultural areas” that has been removed by wheat‑only cultivation (Interaction 3).  

**IMPACTED METRICS**  
- **soil_organic_carbon:** increase (direction ↑)  
- **water_retention_capacity:** increase (↑)  
- **species_richness (s

In [61]:
IMPORTANT_FIELDS = {
    "soil_organic_carbon": "soil organic carbon percentage",
    "rainfall": "rainfall level or annual rainfall",
    "land_use": "current land use",
    "region": "region or climate type"
}

def find_missing_fields(profile):
    missing = []

    data = profile.model_dump()

    for field, description in IMPORTANT_FIELDS.items():
        if data.get(field) is None:
            missing.append((field, description))

    return missing

In [62]:
incomplete_profile = EnvironmentalProfile(
    land_use="agriculture"
)

print(find_missing_fields(incomplete_profile))

[('soil_organic_carbon', 'soil organic carbon percentage'), ('rainfall', 'rainfall level or annual rainfall'), ('region', 'region or climate type')]


In [63]:
def generate_clarifying_question(profile):

    missing = find_missing_fields(profile)

    if not missing:
        return None

    descriptions = [item[1] for item in missing]

    prompt = f"""
You are an environmental assessment assistant.

The user has provided some environmental information, but important information is missing.

Missing information:
{descriptions}

Ask the user concise and friendly follow-up questions.

Rules:
- Ask only for the missing information.
- Do not recommend interventions yet.
- Tell the user they can say "unknown" if they do not know a value.
- Keep the response short.
"""

    response = llm.invoke(prompt)

    return response.content

In [64]:
question = generate_clarifying_question(incomplete_profile)

print(question)

Could you let me know:

1. The soil organic carbon % (or “unknown” if you don’t have it)  
2. The typical rainfall level or annual rainfall amount (or “unknown”)  
3. The region or climate type of the site (or “unknown”)  

Thanks!


In [65]:
def assess_profile(profile):

    missing = find_missing_fields(profile)

    if missing:
        return {
            "status": "needs_more_information",
            "response": generate_clarifying_question(profile)
        }

    analysis, docs = analyze_environmental_profile(profile)

    return {
        "status": "ready",
        "response": analysis
    }

In [66]:
test_profile = EnvironmentalProfile(
    land_use="wheat farm"
)

result = assess_profile(test_profile)

print(result["status"])
print()
print(result["response"])

needs_more_information

Could you let me know:

1. The soil organic carbon % (or “unknown” if you don’t have it)  
2. The typical rainfall level or annual rainfall amount (or “unknown”)  
3. The region or climate type of the site (or “unknown”)  

Thanks!


In [67]:
result = assess_profile(profile)

print(result["status"])

ready


In [68]:
extractor_llm = llm.with_structured_output(EnvironmentalProfile)

def extract_environmental_profile(user_text):

    prompt = f"""
Extract environmental information from the user's message.

USER MESSAGE:
{user_text}

Rules:
- Extract information explicitly stated by the user.
- You may make simple, obvious category mappings.
- Do NOT invent scientific measurements.
- If something is unknown or not mentioned, return null.

Useful mappings:
- "farm", "wheat farm", "crop field" -> land_use = "agriculture"
- "forest" -> land_use = "forest"
- "pasture" -> land_use = "pasture"
- "semi arid" -> region = "semi-arid"
- "soil carbon is 0.3%" -> soil_organic_carbon = 0.3
- "biodiversity is declining" -> biodiversity_trend = "declining"

Important:
- Do NOT convert general biodiversity decline into habitat_diversity.
- Do NOT convert general biodiversity decline into species_richness.
- Only set habitat_diversity if the user specifically talks about habitat diversity.
- Only set species_richness if the user specifically talks about species richness.
"""

    return extractor_llm.invoke(prompt)

In [69]:
user_message = """
I have a wheat farm in a semi-arid region.
Rainfall is low and my soil organic carbon is around 0.3%.
"""

extracted_profile = extract_environmental_profile(user_message)

print(extracted_profile.model_dump())

{'soil_organic_carbon': 0.3, 'soil_ph': None, 'soil_moisture': None, 'rainfall': 'low', 'temperature': None, 'land_use': 'agriculture', 'crop': None, 'biodiversity_trend': None, 'species_richness': None, 'habitat_diversity': None, 'pollution': None, 'deforestation': None, 'region': 'semi-arid'}


In [70]:
def process_user_message(user_text):

    profile = extract_environmental_profile(user_text)

    print("Extracted Environmental Profile:")
    print(profile.model_dump(exclude_none=True))
    print()

    result = assess_profile(profile)

    return profile, result

In [71]:
profile_test, result = process_user_message(
    "Biodiversity is declining on my wheat farm."
)

print("STATUS:", result["status"])
print()
print(result["response"])

Extracted Environmental Profile:
{'land_use': 'agriculture', 'biodiversity_trend': 'declining'}

STATUS: needs_more_information

Could you let me know:

1. The soil organic carbon % (or “unknown” if you don’t have it)  
2. The typical rainfall amount or annual rainfall for the site (or “unknown”)  
3. The region or climate type (e.g., temperate, tropical, arid, etc.) (or “unknown”)  


In [72]:
profile_test2, result2 = process_user_message(
    """
    My wheat farm is in a semi-arid region.
    Soil organic carbon is 0.3%.
    Rainfall is low.
    """
)

print("STATUS:", result2["status"])
print()
print(result2["response"])

Extracted Environmental Profile:
{'soil_organic_carbon': 0.3, 'rainfall': 'low', 'land_use': 'agriculture', 'region': 'semi-arid'}

STATUS: ready

**ENVIRONMENTAL DIAGNOSIS**  
The semi‑arid agricultural landscape described (soil organic carbon = 0.3 % ≈ very low, low rainfall, dominant cropland) is characterised by a set of tightly coupled stressors:

| Stressor | Evidence‑based implication |
|----------|-----------------------------|
| **Very low soil organic carbon (SOC)** | SOC is a recognised indicator of land‑and‑soil degradation and of the capacity of terrestrial ecosystems to sequester carbon (Laurenz & Lal 2016, Source 4, p. 77). |
| **Low precipitation** | In semi‑arid zones limited rainfall restricts primary productivity, which in turn limits the amount of plant litter that can become SOC. |
| **Intensive agricultural land‑use** | Global assessments show that conversion to agriculture has reduced biodiversity by 11‑14 % and drives desertification and land‑degradation (IPCC S

In [73]:
conversation_state = {
    "profile": EnvironmentalProfile(),
    "history": []
}

In [74]:
def merge_profiles(old_profile, new_profile):
    
    old_data = old_profile.model_dump()
    new_data = new_profile.model_dump()

    for key, value in new_data.items():
        if value is not None:
            old_data[key] = value

    return EnvironmentalProfile(**old_data)

In [75]:
def chat(user_message):

    global conversation_state

    # Save user message
    conversation_state["history"].append({
        "role": "user",
        "content": user_message
    })

    # Extract environmental information from new message
    new_profile = extract_environmental_profile(user_message)

    # Merge with previous information
    conversation_state["profile"] = merge_profiles(
        conversation_state["profile"],
        new_profile
    )

    profile = conversation_state["profile"]

    print("CURRENT ENVIRONMENTAL PROFILE:")
    print(profile.model_dump(exclude_none=True))
    print()

    # Check if enough information exists
    missing = find_missing_fields(profile)

    if missing:
        response = generate_clarifying_question(profile)

    else:
        diagnosis, docs = analyze_environmental_profile(profile)

        recommendations, rec_docs = generate_recommendations(
            profile,
            diagnosis
        )

        response = f"""
{diagnosis}

==================================================
RECOMMENDATIONS
==================================================

{recommendations}
"""

    conversation_state["history"].append({
        "role": "assistant",
        "content": response
    })

    return response

In [76]:
def reset_conversation():

    global conversation_state

    conversation_state = {
        "profile": EnvironmentalProfile(),
        "history": []
    }

    print("Conversation reset.")

In [77]:
reset_conversation()

Conversation reset.


In [49]:
response = chat(
    "Biodiversity is declining on my wheat farm."
)

print(response)

CURRENT ENVIRONMENTAL PROFILE:
{'land_use': 'agriculture', 'crop': 'wheat', 'habitat_diversity': 'declining'}

Could you let me know:

1. The soil organic carbon % (or “unknown” if you don’t have it)  
2. The typical rainfall level or annual rainfall amount (or “unknown”)  
3. The region or climate type of the site (or “unknown”)  

Thanks!


In [50]:
response = chat(
    "My soil organic carbon is around 0.3% and rainfall is low."
)

print(response)

CURRENT ENVIRONMENTAL PROFILE:
{'soil_organic_carbon': 0.3, 'rainfall': 'low', 'land_use': 'agriculture', 'crop': 'wheat', 'habitat_diversity': 'declining'}

Could you let me know the region or climate type for the area you’re assessing? (If you’re not sure, just reply “unknown.”)


In [51]:
response = chat(
    "The farm is in a semi-arid region."
)

print(response)

CURRENT ENVIRONMENTAL PROFILE:
{'soil_organic_carbon': 0.3, 'rainfall': 'low', 'land_use': 'agriculture', 'crop': 'wheat', 'habitat_diversity': 'declining', 'region': 'semi-arid'}


**ENVIRONMENTAL DIAGNOSIS**  
The semi‑arid wheat‑producing landscape is characterised by:

* **Very low soil organic carbon (SOC = 0.3 % / t ha)** – a proxy for depleted carbon stocks and poor soil structure.  
* **Low rainfall** – limits water availability for crops and native vegetation.  
* **Intensive agricultural land‑use** – the dominant land cover is cultivated wheat fields.  
* **Declining habitat‑diversity** – the mosaic of natural patches, field margins and refugia is shrinking.  

Together these attributes point to a **high risk of soil degradation, reduced ecosystem regulation (climate, water, pollination) and accelerated biodiversity loss**.

---

### MULTI‑METRIC INTERACTIONS  

| # | Interaction (variables) | Mechanism & Expected Outcome | Evidence |
|---|--------------------------|---------

In [81]:
from pydantic import BaseModel
from typing import List

class GroundingCheck(BaseModel):
    grounded: bool
    unsupported_claims: List[str]
    explanation: str

grounding_llm = llm.with_structured_output(
    GroundingCheck,
    method="json_schema"
)

print("Grounding checker ready!")

Grounding checker ready!


In [82]:
def check_groundedness(answer, docs):

    evidence = format_evidence(docs)

    prompt = f"""
You are a strict scientific evidence verifier.

SCIENTIFIC EVIDENCE:
{evidence}

ANSWER TO VERIFY:
{answer}

Check whether the claims in the answer are supported by the evidence.

Rules:
- Use ONLY the supplied evidence.
- Do not use outside knowledge.
- Specific numbers, percentages, species names, exact time estimates,
  or exact intervention details need direct evidence.
- If something is only a reasonable inference but not directly supported,
  mark it unsupported unless clearly labelled as inference.
- Be strict.

Return:
- grounded: true or false
- unsupported_claims: list
- explanation: short explanation
"""

    return grounding_llm.invoke(prompt)

In [83]:
grounding_result = check_groundedness(
    recommendations,
    recommendation_docs
)

print("GROUNDED:", grounding_result.grounded)

print("\nUNSUPPORTED CLAIMS:")
for claim in grounding_result.unsupported_claims:
    print("-", claim)

print("\nEXPLANATION:")
print(grounding_result.explanation)

GROUNDED: False

UNSUPPORTED CLAIMS:
- Agro‑forestry strips increase soil organic carbon and water‑holding capacity (specific mechanism)
- Agro‑forestry strips restore habitat and increase species richness
- Agro‑forestry is identified as an effective nature‑based solution for semi‑arid croplands (high confidence claim)
- Medium‑term (1–3 years) time horizon for tree/shrub establishment
- Legume cover crops supply nitrogen‑rich organic matter without extra irrigation and raise SOC
- Legume residues improve soil structure and water‑holding capacity
- Legume root exudates diversify the rhizosphere and stimulate microbial communities
- Reduced tillage preserves soil fauna habitats and limits disturbance‑induced SOC losses (specific mechanism)
- Cover crops are listed as integrated pest‑ and nutrient‑management options in the cited source
- Buffer strips increase pollinator abundance
- Buffer strips decrease soil erosion rate
- Buffer strips improve overall biodiversity intactness
- Native

In [84]:
def revise_to_grounded(answer, docs, grounding_result):

    evidence = format_evidence(docs)

    unsupported = "\n".join(
        f"- {claim}"
        for claim in grounding_result.unsupported_claims
    )

    prompt = f"""
You are a scientific answer editor.

ORIGINAL ANSWER:
{answer}

SCIENTIFIC EVIDENCE:
{evidence}

UNSUPPORTED CLAIMS FOUND:
{unsupported}

Rewrite the answer so that it is scientifically grounded.

Rules:
- Remove unsupported numerical values.
- Remove unsupported species names.
- Remove unsupported exact time estimates.
- Remove unsupported mechanisms.
- Do not invent new scientific claims.
- Keep recommendations only when they are supported by the evidence.
- If something is a reasonable inference but not directly supported,
  clearly label it as "inference".
- Preserve useful structure:
  Recommendation
  Why it works
  Impacted metrics
  Time horizon
  Confidence
  Evidence
- Confidence should describe EVIDENCE STRENGTH:
    High = directly supported by multiple retrieved sources
    Medium = supported but evidence is limited
    Low = mainly inference
"""

    response = llm.invoke(prompt)

    return response.content

In [85]:
grounded_recommendations = revise_to_grounded(
    recommendations,
    recommendation_docs,
    grounding_result
)

print(grounded_recommendations)

**Recommendation 1 – Integrate agro‑forestry strips (nitrogen‑fixing trees or drought‑tolerant shrubs) into the wheat field matrix**  

**Why it works** –  
*Inference*: Woody species add leaf litter and fine roots that can increase the amount of organic material returned to the soil, which is known to raise soil organic carbon (SOC) and improve water‑holding capacity. The presence of trees and shrubs also creates structural heterogeneity that can support a wider range of soil microbes and above‑ground organisms, thereby increasing habitat availability.  

**Impacted metrics**  
- soil_organic_carbon: increase (↑) – inference  
- water_retention_capacity: increase (↑) – inference  
- species_richness (soil & above‑ground): increase (↑) – inference  
- climate_regulation (soil C sequestration): improve (↑) – inference  

**Time horizon** – Benefits begin to appear as the woody plants become established (generally within the first few growing seasons).  

**Confidence** – **High** – Agro

In [86]:
second_check = check_groundedness(
    grounded_recommendations,
    recommendation_docs
)

print("GROUNDED:", second_check.grounded)

print("\nUNSUPPORTED CLAIMS:")
for claim in second_check.unsupported_claims:
    print("-", claim)

print("\nEXPLANATION:")
print(second_check.explanation)

GROUNDED: False

UNSUPPORTED CLAIMS:
- Woody species add leaf litter and fine roots that can increase organic material returned to the soil, raising soil organic carbon and improving water‑holding capacity
- The presence of trees and shrubs creates structural heterogeneity that supports a wider range of soil microbes and above‑ground organisms
- Agro‑forestry is listed among nature‑based solutions that improve ecosystem functions in semi‑arid agricultural landscapes
- Benefits begin to appear as woody plants become established (generally within the first few growing seasons)
- Legume residues are rich in nitrogen and organic matter and add carbon to the soil, improving structure and water retention
- Legume root exudates diversify the rhizosphere and support a more diverse microbial community
- Reduced or no‑till minimizes disturbance of soil aggregates and habitats of soil fauna, helping retain SOC and biodiversity
- Impacted metrics such as increased soil_organic_carbon, water_retent

In [87]:
def retrieve_intervention_evidence(intervention, profile, k=6):

    profile_text = json.dumps(
        profile.model_dump(exclude_none=True),
        indent=2
    )

    query = f"""
Scientific evidence about {intervention}.

Environmental context:
{profile_text}

Retrieve evidence about:
- biodiversity effects
- soil organic carbon
- soil health
- water or moisture
- habitat diversity
- land degradation
"""

    return vectorstore.similarity_search(query, k=k)

In [88]:
agro_docs = retrieve_intervention_evidence(
    "agroforestry in agricultural landscapes",
    profile,
    k=8
)

for i, doc in enumerate(agro_docs, 1):
    print("=" * 70)
    print("RESULT", i)
    print("Source:", doc.metadata.get("source_file"))
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content[:800])
    print()

RESULT 1
Source: agroforestry_primer.pdf
Page: 18
and ecologically sustainable form of agriculture and land use. But 
now agroforestry is suddenly at centre stage. It is promoted as a land-
use strategy to support climate change mitigation and climate change 
adaptation, biodiversity conservation, sustainable agriculture and other 
goals. Many organizations recommend or use it as a tool for restoring 
ecosystems, not only agricultural ones, but also forest landscapes.

RESULT 2
Source: agroforestry_primer.pdf
Page: 16
INTRODUCTION
15
AGROFORESTRY: A PRIMER
1
C
onventional agriculture is very productive. But high 
productivity comes at a cost: soil that is depleted or 
eroded, watercourses that are polluted or drying up, and a 
food system that produces 20–40% of greenhouse gas emissions. 
Many people now agree that we urgently need to transform the 
food system, including agriculture. Agroforestry, as a nature-based 
approach to production and land use, will play an important role in 


In [89]:
cover_crop_docs = retrieve_intervention_evidence(
    "cover crops soil organic carbon soil biodiversity sustainable agriculture",
    profile,
    k=8
)

tillage_docs = retrieve_intervention_evidence(
    "reduced tillage no till soil organic carbon soil biodiversity land degradation",
    profile,
    k=8
)

habitat_docs = retrieve_intervention_evidence(
    "field margins natural habitat biodiversity pollinators agricultural land",
    profile,
    k=8
)

In [90]:
for i, doc in enumerate(cover_crop_docs, 1):
    print("=" * 70)
    print("RESULT", i)
    print("Source:", doc.metadata.get("source_file"))
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content[:700])
    print()

RESULT 1
Source: fao_soil_carbon.pdf
Page: 77
B. C., Trumbore, S. E. & Gleixner, G. 2015. Plant diversity increases soil microbial 
activity and soil carbon storage. Nature Communications, 6: 6707. 
Laurenz, K. & Lal, R. 2016. Soil Organic Carbon - An appropriate Indicator to Monitor 
Trends of Land and Soil Degradation within the SDG Framework? Dessau-Roßlau: 
Umweltbundesamt.

RESULT 2
Source: fao_soil_carbon.pdf
Page: 7
VI
SOIL ORGANIC CARBON  the hidden potential
EXECUTIVE SUMMARY
In the presence of climate change, land degradation and biodiversity loss, soils have 
become one of the most vulnerable resources in the world. Soils are a major carbon 
reservoir containing more carbon than the atmosphere and terrestrial vegetation 
combined. Soil organic carbon (SOC) is dynamic, however, and anthropogenic 
impacts on soil can turn it into either a net sink or a net source of GHGs. Enormous 
scientific progress has been achieved in understanding and explaining SOC dynamics. 
Yet, protec

In [91]:
for i, doc in enumerate(tillage_docs, 1):
    print("=" * 70)
    print("RESULT", i)
    print("Source:", doc.metadata.get("source_file"))
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content[:700])
    print()

RESULT 1
Source: fao_soil_carbon.pdf
Page: 77
B. C., Trumbore, S. E. & Gleixner, G. 2015. Plant diversity increases soil microbial 
activity and soil carbon storage. Nature Communications, 6: 6707. 
Laurenz, K. & Lal, R. 2016. Soil Organic Carbon - An appropriate Indicator to Monitor 
Trends of Land and Soil Degradation within the SDG Framework? Dessau-Roßlau: 
Umweltbundesamt.

RESULT 2
Source: fao_soil_carbon.pdf
Page: 61
50
SOIL ORGANIC CARBON  the hidden potential
Figure 13 · Suggested and dissuaded management strategies for soil carbon sequestration 
and their impact on food productivity and climate change mitigation and adaptation.
Colours indicate good (green) and bad (red) practices. Partially adapted and modified 
from Ogle et al., 2014, and Descheemaeker et al., 2016
ADAPTATION MITIGATION FOOD
PRODUCTIVITY
Reforestation/afforestation of arable land
Han et al., 2016
Conservation/reduced tillage1
       Haddaway et al., 2016 - Mangalassery et al., 2015
Crop rotations
Raphael et

In [92]:
for i, doc in enumerate(habitat_docs, 1):
    print("=" * 70)
    print("RESULT", i)
    print("Source:", doc.metadata.get("source_file"))
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content[:700])
    print()

RESULT 1
Source: fao_soil_carbon.pdf
Page: 77
B. C., Trumbore, S. E. & Gleixner, G. 2015. Plant diversity increases soil microbial 
activity and soil carbon storage. Nature Communications, 6: 6707. 
Laurenz, K. & Lal, R. 2016. Soil Organic Carbon - An appropriate Indicator to Monitor 
Trends of Land and Soil Degradation within the SDG Framework? Dessau-Roßlau: 
Umweltbundesamt.

RESULT 2
Source: fao_soil_carbon.pdf
Page: 24
where it is believed to have longer residence times (Rumpel and Kögel-Knabner, 2011).
2.2.2 · SOIL BIODIVERSITY LOSSES
Losses in soil biodiversity have been demonstrated to affect multiple ecosystem 
functions including decomposition of SOC, nutrient retention and nutrient cycling 
(FAO and ITPS, 2015). Poor land-management practices and environmental change 
are affecting belowground communities globally, and the resulting declines in soil 
biodiversity reduce and impair these benefits (Figure 3) (Wall et al., 2015).
Figure 3 · Impact of land use decisions on soil 

In [93]:
def is_reference_chunk(doc):
    text = doc.page_content.lower()

    reference_signals = [
        "references",
        "bibliography",
        "doi:",
        "et al."
    ]

    count = sum(signal in text for signal in reference_signals)

    return count >= 2

In [94]:
def retrieve_clean_evidence(query, k=12, final_k=6):

    docs = vectorstore.similarity_search(query, k=k)

    clean_docs = [
        doc for doc in docs
        if not is_reference_chunk(doc)
    ]

    return clean_docs[:final_k]

In [95]:
cover_docs = retrieve_clean_evidence(
    """
    cover cropping crop rotation soil organic carbon
    sustainable soil management biodiversity agriculture
    """
)

for i, doc in enumerate(cover_docs, 1):
    print("=" * 70)
    print("RESULT", i)
    print("Source:", doc.metadata.get("source_file"))
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content[:700])
    print()

RESULT 1
Source: fao_soil_carbon.pdf
Page: 61
50
SOIL ORGANIC CARBON  the hidden potential
Figure 13 · Suggested and dissuaded management strategies for soil carbon sequestration 
and their impact on food productivity and climate change mitigation and adaptation.
Colours indicate good (green) and bad (red) practices. Partially adapted and modified 
from Ogle et al., 2014, and Descheemaeker et al., 2016
ADAPTATION MITIGATION FOOD
PRODUCTIVITY
Reforestation/afforestation of arable land
Han et al., 2016
Conservation/reduced tillage1
       Haddaway et al., 2016 - Mangalassery et al., 2015
Crop rotations
Raphael et al., 2016
Cover cropping
Poeplau and Don, 2015
Organic farming2
Skinner et al., 2014
Balanced combined applications of chemical

RESULT 2
Source: fao_soil_carbon.pdf
Page: 25
14
SOIL ORGANIC CARBON  the hidden potential
With ongoing losses in belowground microbial diversity, understanding relationships 
between soil biodiversity and C cycling is critical for projecting how the l

In [96]:
tillage_docs = retrieve_clean_evidence(
    """
    conservation tillage reduced tillage soil organic carbon
    sustainable agriculture soil biodiversity
    """
)

for i, doc in enumerate(tillage_docs, 1):
    print("=" * 70)
    print("RESULT", i)
    print("Source:", doc.metadata.get("source_file"))
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content[:700])
    print()

RESULT 1
Source: fao_soil_carbon.pdf
Page: 61
50
SOIL ORGANIC CARBON  the hidden potential
Figure 13 · Suggested and dissuaded management strategies for soil carbon sequestration 
and their impact on food productivity and climate change mitigation and adaptation.
Colours indicate good (green) and bad (red) practices. Partially adapted and modified 
from Ogle et al., 2014, and Descheemaeker et al., 2016
ADAPTATION MITIGATION FOOD
PRODUCTIVITY
Reforestation/afforestation of arable land
Han et al., 2016
Conservation/reduced tillage1
       Haddaway et al., 2016 - Mangalassery et al., 2015
Crop rotations
Raphael et al., 2016
Cover cropping
Poeplau and Don, 2015
Organic farming2
Skinner et al., 2014
Balanced combined applications of chemical

RESULT 2
Source: fao_soil_carbon.pdf
Page: 25
14
SOIL ORGANIC CARBON  the hidden potential
With ongoing losses in belowground microbial diversity, understanding relationships 
between soil biodiversity and C cycling is critical for projecting how the l

In [97]:
import re

def is_reference_chunk(doc):
    text = doc.page_content.lower()

    et_al_count = text.count("et al.")
    year_count = len(re.findall(r"\b(19|20)\d{2}\b", text))

    if "references" in text[:200]:
        return True

    if et_al_count >= 3 and year_count >= 4:
        return True

    return False

In [98]:
def retrieve_clean_evidence(query, k=15, final_k=6):

    docs = vectorstore.similarity_search(query, k=k)

    clean_docs = [
        doc for doc in docs
        if not is_reference_chunk(doc)
    ]

    return clean_docs[:final_k]

In [99]:
cover_docs = retrieve_clean_evidence(
    """
    cover cropping crop rotation soil organic carbon
    sustainable soil management biodiversity agriculture
    """
)

for i, doc in enumerate(cover_docs, 1):
    print("=" * 70)
    print("RESULT", i)
    print("Source:", doc.metadata.get("source_file"))
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content[:700])
    print()

RESULT 1
Source: fao_soil_carbon.pdf
Page: 25
14
SOIL ORGANIC CARBON  the hidden potential
With ongoing losses in belowground microbial diversity, understanding relationships 
between soil biodiversity and C cycling is critical for projecting how the loss of diversity 
under continued environmental alteration by humans will impact global C cycling 
processes (De Graaf et al., 2015).
Current research indicates that soil biodiversity can be maintained and partially 
restored if managed sustainably. Promoting the ecological complexity and robustness of 
soil biodiversity through improved management practices represents an underutilized 
resource with the ability to ultimately improve human health (Figure 3) (Wall et al., 
2015). For sustai

RESULT 2
Source: fao_soil_carbon.pdf
Page: 57
46
SOIL ORGANIC CARBON  the hidden potential
5 · SOC MANAGEMENT 
FOR SUSTAINABLE 
FOOD PRODUCTION 
AND CLIMATE CHANGE 
MITIGATION  
AND ADAPTATION  
©FAO/Daniel Hayduk

RESULT 3
Source: fao_soil_carbon.pdf


In [100]:
def build_recommendation_evidence(profile):

    agro_docs = retrieve_clean_evidence(
        "agroforestry biodiversity conservation climate regulation habitat connectivity agriculture",
        k=15,
        final_k=5
    )

    cover_docs = retrieve_clean_evidence(
        "cover cropping crop rotation soil organic carbon sustainable soil management",
        k=15,
        final_k=5
    )

    tillage_docs = retrieve_clean_evidence(
        "conservation reduced tillage soil organic carbon sustainable agriculture",
        k=15,
        final_k=5
    )

    # combine and remove duplicates
    all_docs = agro_docs + cover_docs + tillage_docs

    unique_docs = []
    seen = set()

    for doc in all_docs:
        key = (
            doc.metadata.get("source_file"),
            doc.metadata.get("page"),
            doc.page_content[:100]
        )

        if key not in seen:
            seen.add(key)
            unique_docs.append(doc)

    return unique_docs

In [104]:
def generate_final_recommendations(profile, diagnosis):

    profile_text = json.dumps(
        profile.model_dump(exclude_none=True),
        indent=2
    )

    docs = build_recommendation_evidence(profile)
    evidence = format_evidence(docs)

    prompt = f"""
You are a conservative evidence-grounded environmental decision-support system.

ENVIRONMENTAL PROFILE:
{profile_text}

ENVIRONMENTAL DIAGNOSIS:
{diagnosis}

RETRIEVED SCIENTIFIC EVIDENCE:
{evidence}

Generate exactly 3 recommendations based only on interventions found
in the retrieved evidence.

CRITICAL RULES:

1. Never add a mechanism unless the retrieved evidence directly states it.
2. Never invent percentages, species, field sizes or quantitative improvements.
3. Never claim exact improvement amounts.
4. Do not say cover crops improve water retention unless retrieved evidence says so.
5. Do not say reduced tillage improves infiltration unless retrieved evidence says so.
6. Do not claim biodiversity effects for a practice unless directly supported.
7. Clearly separate direct evidence from reasoning.

For each recommendation use:

RECOMMENDATION:
...

DIRECT EVIDENCE:
State ONLY what the supplied documents directly support.

RELEVANCE TO THIS SITE:
This may contain reasoning based on the user's environmental profile.
If it is not directly proven by the evidence, start the sentence with:
"Inference:"

TARGETED METRICS:
Only list metrics directly connected to the intervention by supplied evidence.
If uncertain, write "not established by retrieved evidence".

TIME HORIZON:
Give Short / Medium / Long as a PLANNING ESTIMATE only.
Write exactly:
"Planning estimate: ...; not established by retrieved evidence."

CONFIDENCE:
High / Medium / Low confidence IN THE EVIDENCE SUPPORT,
not confidence that the environmental outcome will definitely occur.

EVIDENCE:
- filename and page

Finally:

PRIORITY ORDER:
Rank the recommendations for this user's profile.
Clearly label this ranking as "Decision-support judgement",
not a scientifically proven universal ranking.
"""

    response = llm.invoke(prompt)

    return response.content, docs

In [106]:
def check_groundedness(answer, docs):

    evidence = format_evidence(docs)

    prompt = f"""
You are a strict scientific evidence verifier.

SCIENTIFIC EVIDENCE:
{evidence}

ANSWER:
{answer}

Check SCIENTIFIC FACTUAL CLAIMS against the supplied evidence.

Rules:
- Use ONLY supplied evidence.
- Specific mechanisms require direct support.
- Numerical claims require direct support.
- Species-specific claims require direct support.
- Clearly labelled "Inference:" statements are allowed as reasoning,
  but must not be presented as established scientific evidence.
- Clearly labelled "Planning estimate:" statements are decision-support
  judgements and should NOT make the answer ungrounded.
- Clearly labelled "Decision-support judgement" rankings should NOT make
  the answer ungrounded.
- If an inference is presented as fact, mark it unsupported.
- Be strict about scientific claims.

Return:
- grounded: true or false
- unsupported_claims: list
- explanation: short explanation
"""

    return grounding_llm.invoke(prompt)

In [107]:
final_recommendations, final_docs = generate_final_recommendations(
    profile,
    analysis
)

print(final_recommendations)

**RECOMMENDATION 1**  
Introduce trees (agroforestry elements) into the wheat field to create on‑farm woody habitat.

**DIRECT EVIDENCE**  
- *agroforestry_primer.pdf*, p. 82: “Trees‑on‑farms contribute directly to the conservation of biodiversity by increasing agrobiodiversity… providing foraging or breeding opportunities for wild or farmland‑adapted animals.”  
- *agroforestry_primer.pdf*, p. 11: “Trees on farms… provide habitat for biodiversity, regenerate soil and water resources and suck carbon dioxide out of the atmosphere.”  
- *agroforestry_primer.pdf*, p. 18: “Agroforestry is promoted as a land‑use strategy to support climate change mitigation and biodiversity conservation.”

**RELEVANCE TO THIS SITE**  
Inference: Adding trees will add structural diversity to a monoculture wheat system, thereby creating habitat, increasing on‑farm biodiversity and contributing to carbon sequestration, which directly addresses the low SOC and habitat‑loss risks identified in the diagnosis.

**

In [108]:
final_grounding = check_groundedness(
    final_recommendations,
    final_docs
)

print("GROUNDED:", final_grounding.grounded)

print("\nUNSUPPORTED CLAIMS:")
for claim in final_grounding.unsupported_claims:
    print("-", claim)

print("\nEXPLANATION:")
print(final_grounding.explanation)

GROUNDED: False

UNSUPPORTED CLAIMS:
- The claim that reduced‑tillage or no‑till management will increase soil organic carbon in the wheat field is not directly supported by the cited evidence, which only notes that tillage intensity affects SOC without specifying the effect direction.
- The claim that diversified crop rotations (e.g., including legumes) will raise soil organic carbon by providing additional carbon inputs is not directly supported by the cited evidence, which merely mentions crop rotations in relation to soil organic matter without demonstrating a causal increase in SOC.

EXPLANATION:
While the agroforestry recommendations are directly backed by the provided excerpts, the statements about reduced‑tillage/no‑till and diversified rotations improving SOC are inferred rather than explicitly evidenced in the cited sources.


In [109]:
def generate_submission_recommendations(profile, diagnosis):

    profile_text = json.dumps(
        profile.model_dump(exclude_none=True),
        indent=2
    )

    docs = build_recommendation_evidence(profile)
    evidence = format_evidence(docs)

    prompt = f"""
You are a highly conservative scientific environmental decision-support system.

ENVIRONMENTAL PROFILE:
{profile_text}

ENVIRONMENTAL DIAGNOSIS:
{diagnosis}

SCIENTIFIC EVIDENCE:
{evidence}

Generate exactly 3 practical recommendations.

STRICT GROUNDING RULES:

1. State only what the retrieved evidence directly supports.
2. Never turn "listed as a management strategy" into
   "will increase", "will improve", or "will reduce".
3. For conservation/reduced tillage:
   only state that FAO lists it as a soil-carbon management strategy
   unless stronger evidence is explicitly present.
4. For crop rotations or cover cropping:
   only state that FAO lists them as soil-carbon management strategies
   unless stronger evidence is explicitly present.
5. Do NOT claim crop rotations increase SOC unless directly supported.
6. Do NOT claim reduced tillage increases SOC unless directly supported.
7. Do NOT introduce legumes unless the evidence specifically supports legumes.
8. Agroforestry claims must also remain limited to what the retrieved
   agroforestry evidence directly states.
9. Do not invent percentages, species, field sizes or numerical improvements.
10. Clearly label reasoning that goes beyond direct evidence as "Inference:".

For each recommendation provide:

RECOMMENDATION:
...

DIRECT EVIDENCE:
...

RELEVANCE TO THIS SITE:
If this contains reasoning beyond the evidence, begin with "Inference:".

TARGETED METRICS:
Only metrics directly supported by evidence.
Do not use arrows showing improvement unless direction is directly supported.

TIME HORIZON:
Planning estimate: Short / Medium / Long;
not established by retrieved evidence.

CONFIDENCE:
High / Medium / Low confidence in evidence support.

EVIDENCE:
- filename and page

Finally:

PRIORITY ORDER:
Label this explicitly as:
"Decision-support judgement, not a scientifically proven universal ranking."
"""

    response = llm.invoke(prompt)

    return response.content, docs

In [110]:
submission_recommendations, submission_docs = generate_submission_recommendations(
    profile,
    analysis
)

print(submission_recommendations)

**RECOMMENDATION 1**  
Adopt a reduced‑tillage (conservation‑tillage) system.

**DIRECT EVIDENCE**  
FAO’s “Soil Organic Carbon – the hidden potential” identifies reduced tillage as a soil‑carbon management strategy for sustainable food production and climate‑change mitigation. (Source 6, page 57)

**RELEVANCE TO THIS SITE**  
*Inference:* Reduced tillage is listed as a management option that can help maintain or improve soil organic carbon, which is currently very low (0.3 %). Applying this option may mitigate the feedback loop of low SOC ↔ poor water retention ↔ low productivity described in the diagnosis.

**TARGETED METRICS**  
- Soil organic carbon (SOC)  
- Water‑holding capacity (implicit to SOC)  

**TIME HORIZON**  
Medium (implementation in the next cropping season; effects observable over several seasons)

**CONFIDENCE**  
High (directly listed by FAO)

**EVIDENCE**  
- fao_soil_carbon.pdf, p. 57  


---

**RECOMMENDATION 2**  
Introduce diversified crop rotations (e.g., alt

In [111]:
submission_grounding = check_groundedness(
    submission_recommendations,
    submission_docs
)

print("GROUNDED:", submission_grounding.grounded)

print("\nUNSUPPORTED CLAIMS:")
for claim in submission_grounding.unsupported_claims:
    print("-", claim)

print("\nEXPLANATION:")
print(submission_grounding.explanation)

GROUNDED: False

UNSUPPORTED CLAIMS:
- FAO’s “Soil Organic Carbon – the hidden potential” identifies reduced tillage as a soil‑carbon management strategy for sustainable food production and climate‑change mitigation.
- Reduced tillage is listed as a management option that can help maintain or improve soil organic carbon, which is currently very low (0.3%).
- FAO’s “Soil Organic Carbon – the hidden potential” lists crop rotations as a soil‑carbon management strategy.
- Confidence (directly listed by FAO) for the recommendations is stated as high.

EXPLANATION:
The answer cites FAO document evidence for reduced tillage and crop rotations, but the provided excerpts do not contain those statements, and it includes an unsupported numeric SOC value; only the agroforestry claims are supported.


In [112]:
def safe_recommendation_output(profile, diagnosis):

    recommendations, docs = generate_submission_recommendations(
        profile,
        diagnosis
    )

    grounding = check_groundedness(
        recommendations,
        docs
    )

    return {
        "recommendations": recommendations,
        "grounded": grounding.grounded,
        "unsupported_claims": grounding.unsupported_claims,
        "grounding_explanation": grounding.explanation,
        "sources": docs
    }

In [113]:
safe_result = safe_recommendation_output(
    profile,
    analysis
)

print("GROUNDING STATUS:", safe_result["grounded"])

print("\nRECOMMENDATIONS:\n")
print(safe_result["recommendations"])

if not safe_result["grounded"]:
    print("\n⚠ Evidence verification notes:")
    for claim in safe_result["unsupported_claims"]:
        print("-", claim)

GROUNDING STATUS: False

RECOMMENDATIONS:

**RECOMMENDATION 1**  
Adopt agroforestry elements (e.g., planting native tree species on field margins or within the wheat field) to increase agrobiodiversity and provide habitat for wildlife.

**DIRECT EVIDENCE**  
- *agroforestry_primer.pdf*, p. 82 – “Trees‑on‑farms contribute directly to the conservation of biodiversity… increasing agrobiodiversity, providing foraging or breeding opportunities for wild or farmland‑adapted animals.”  
- *agroforestry_primer.pdf*, p. 11 – Agroforestry “provides habitat for biodiversity, regenerates soil and water resources.”  
- *agroforestry_primer.pdf*, p. 18 – Agroforestry is promoted as a land‑use strategy to support biodiversity conservation.

**RELEVANCE TO THIS SITE**  
*Inference:* Introducing trees adds structural diversity to a monoculture wheat system, which can counter the identified loss of natural habitat and functional biodiversity in a semi‑arid, low‑SOC context.

**TARGETED METRICS**  
- Ext

In [114]:
conversation_state = {
    "profile": EnvironmentalProfile(),
    "history": []
}

In [115]:
def biodiversity_chatbot(user_message):

    global conversation_state

    # 1. Save user message
    conversation_state["history"].append({
        "role": "user",
        "content": user_message
    })

    # 2. Extract environmental information
    new_profile = extract_environmental_profile(user_message)

    # 3. Merge with previous conversation information
    conversation_state["profile"] = merge_profiles(
        conversation_state["profile"],
        new_profile
    )

    profile = conversation_state["profile"]

    # 4. Check missing information
    missing = find_missing_fields(profile)

    if missing:
        response = generate_clarifying_question(profile)

        conversation_state["history"].append({
            "role": "assistant",
            "content": response
        })

        return {
            "status": "needs_more_information",
            "profile": profile.model_dump(exclude_none=True),
            "response": response
        }

    # 5. Multi-metric environmental analysis
    diagnosis, diagnosis_docs = analyze_environmental_profile(profile)

    # 6. Generate recommendations + verify grounding
    safe_result = safe_recommendation_output(
        profile,
        diagnosis
    )

    recommendations = safe_result["recommendations"]

    # 7. Build final response
    response = f"""
ENVIRONMENTAL ASSESSMENT
========================

{diagnosis}

RECOMMENDATIONS
===============

{recommendations}
"""

    conversation_state["history"].append({
        "role": "assistant",
        "content": response
    })

    return {
        "status": "complete",
        "profile": profile.model_dump(exclude_none=True),
        "diagnosis": diagnosis,
        "recommendations": recommendations,
        "grounded": safe_result["grounded"],
        "grounding_notes": safe_result["unsupported_claims"],
        "response": response
    }

In [116]:
def reset_chatbot():

    global conversation_state

    conversation_state = {
        "profile": EnvironmentalProfile(),
        "history": []
    }

    print("Chatbot reset.")

In [117]:
reset_chatbot()

Chatbot reset.


In [118]:
result = biodiversity_chatbot(
    "Biodiversity is declining on my wheat farm."
)

print(result["profile"])
print()
print(result["response"])

{'land_use': 'agriculture', 'biodiversity_trend': 'declining'}

Could you let me know:

1. The soil organic carbon % (or “unknown” if you don’t have it)  
2. The typical rainfall amount or annual rainfall for the site (or “unknown”)  
3. The region or climate type (e.g., temperate, tropical, arid, etc.) (or “unknown”)  


In [119]:
result = biodiversity_chatbot(
    "Soil organic carbon is about 0.3% and rainfall is low."
)

print(result["profile"])
print()
print(result["response"])

{'soil_organic_carbon': 0.3, 'rainfall': 'low', 'land_use': 'agriculture', 'biodiversity_trend': 'declining'}

Could you let me know the region or climate type for the area you’re assessing? (If you’re not sure, just reply “unknown.”)


In [120]:
result = biodiversity_chatbot(
    "It is in a semi-arid region."
)

print(result["profile"])
print()
print(result["response"])

{'soil_organic_carbon': 0.3, 'rainfall': 'low', 'land_use': 'agriculture', 'biodiversity_trend': 'declining', 'region': 'semi-arid'}


ENVIRONMENTAL ASSESSMENT

**ENVIRONMENTAL DIAGNOSIS**  
The semi‑arid agricultural landscape described is characterized by:

* **Very low soil organic carbon (SOC = 0.3 % or t ha⁻¹)** – a proxy for degraded soils and reduced capacity to sequester carbon (Source 4).  
* **Low rainfall** – typical of semi‑arid zones and a driver of water stress and desertification (Source 7).  
* **Dominant land‑use: agriculture** – a land‑use change that is the single largest direct driver of terrestrial biodiversity loss (Source 6; Source 7).  
* **Observed declining biodiversity trend** – consistent with global patterns of ecosystem‐extent loss (Source 5) and with the documented reduction of natural habitat within agricultural matrices (Source 3).  

Together these variables point to a **high‑risk system** where soil degradation, water scarcity, and habitat conversion 

In [121]:
print("STATUS:", result["status"])
print("GROUNDING:", result["grounded"])

if result["grounding_notes"]:
    print("\nEvidence verification notes:")
    for note in result["grounding_notes"]:
        print("-", note)

STATUS: complete
GROUNDING: False

Evidence verification notes:
- The claim that FAO‑listed soil‑carbon management practices specifically include reduced tillage, crop‑rotation, and cover‑cropping is not directly supported by the provided excerpts
- The examples given for sustainable soil‑management actions (e.g., minimize disturbance, retain organic residues) are not explicitly mentioned in the cited FAO soil‑carbon sources


In [122]:
class GroundingCheck(BaseModel):
    status: str
    supported_claims: List[str]
    unsupported_claims: List[str]
    explanation: str

In [123]:
grounding_llm = llm.with_structured_output(
    GroundingCheck,
    method="json_schema"
)

In [124]:
def check_groundedness(answer, docs):

    evidence = format_evidence(docs)

    prompt = f"""
You are a scientific evidence verifier.

SCIENTIFIC EVIDENCE:
{evidence}

ANSWER:
{answer}

Evaluate whether the important scientific claims are supported
by the supplied evidence.

IMPORTANT:
- Paraphrasing is allowed.
- Information shown in tables, figures, headings or lists counts as evidence.
- For example, if a document lists "cover cropping",
  "crop rotations", or "conservation/reduced tillage"
  under soil carbon management strategies, then saying that
  the document identifies these as soil-carbon management
  strategies IS supported.
- Do not demand identical wording.
- Clearly labelled inference is allowed but must not be treated
  as direct scientific evidence.
- Numerical claims require direct evidence.
- Specific species, percentages, mechanisms or exact outcomes
  require direct evidence.

Choose exactly one status:

"Grounded"
= essentially all important scientific claims are supported.

"Partially Grounded"
= the main recommendations are supported but some secondary
details or mechanisms are unsupported.

"Not Grounded"
= the main recommendations themselves lack evidence.

Return:
- status
- supported_claims
- unsupported_claims
- explanation
"""

    return grounding_llm.invoke(prompt)

In [125]:
grounding = check_groundedness(
    submission_recommendations,
    submission_docs
)

print("STATUS:", grounding.status)

print("\nSUPPORTED:")
for claim in grounding.supported_claims:
    print("✓", claim)

print("\nUNSUPPORTED:")
for claim in grounding.unsupported_claims:
    print("⚠", claim)

print("\nEXPLANATION:")
print(grounding.explanation)

STATUS: Partially Grounded

SUPPORTED:
✓ Trees on farms increase agrobiodiversity, conserve indigenous tree species, and provide foraging or breeding opportunities for wildlife (Source 1, p.82)
✓ Agroforestry helps address climate‑change adaptation, provides habitat for biodiversity, and regenerates soil and water resources (Source 2, p.11)

UNSUPPORTED:
⚠ FAO’s “Soil Organic Carbon – the hidden potential” (Source 6, p.57) identifies reduced‑tillage as a soil‑carbon management strategy
⚠ FAO’s “Soil Organic Carbon – the hidden potential” (Source 6, p.57) lists crop rotations as a soil‑carbon management strategy

EXPLANATION:
The answer correctly cites evidence from the agroforestry primer that trees increase agrobiodiversity and contribute to climate‑adaptation and soil regeneration, which is directly supported by the provided excerpts. However, the claims that the FAO soil‑carbon document lists reduced‑tillage and diversified crop rotations as management strategies are not substantiat

In [126]:
def safe_recommendation_output(profile, diagnosis):

    recommendations, docs = generate_submission_recommendations(
        profile,
        diagnosis
    )

    grounding = check_groundedness(
        recommendations,
        docs
    )

    return {
        "recommendations": recommendations,
        "grounding_status": grounding.status,
        "supported_claims": grounding.supported_claims,
        "unsupported_claims": grounding.unsupported_claims,
        "grounding_explanation": grounding.explanation,
        "sources": docs
    }

In [127]:
def biodiversity_chatbot(user_message):

    global conversation_state

    conversation_state["history"].append({
        "role": "user",
        "content": user_message
    })

    new_profile = extract_environmental_profile(user_message)

    conversation_state["profile"] = merge_profiles(
        conversation_state["profile"],
        new_profile
    )

    profile = conversation_state["profile"]

    missing = find_missing_fields(profile)

    if missing:
        response = generate_clarifying_question(profile)

        conversation_state["history"].append({
            "role": "assistant",
            "content": response
        })

        return {
            "status": "needs_more_information",
            "profile": profile.model_dump(exclude_none=True),
            "response": response
        }

    diagnosis, diagnosis_docs = analyze_environmental_profile(profile)

    safe_result = safe_recommendation_output(
        profile,
        diagnosis
    )

    response = f"""
### Environmental Assessment

{diagnosis}

---

### Recommendations

{safe_result["recommendations"]}
"""

    conversation_state["history"].append({
        "role": "assistant",
        "content": response
    })

    return {
        "status": "complete",
        "profile": profile.model_dump(exclude_none=True),
        "diagnosis": diagnosis,
        "recommendations": safe_result["recommendations"],
        "grounding_status": safe_result["grounding_status"],
        "grounding_notes": safe_result["unsupported_claims"],
        "response": response
    }

In [128]:
reset_chatbot()

result = biodiversity_chatbot(
    "I have a wheat farm in a semi-arid region. "
    "Rainfall is low and soil organic carbon is 0.3%."
)

print("STATUS:", result["status"])

if result["status"] == "complete":
    print("GROUNDING:", result["grounding_status"])

Chatbot reset.
STATUS: complete
GROUNDING: Grounded


In [130]:
%%writefile .gitignore
.env
__pycache__/
.ipynb_checkpoints/

Writing .gitignore


In [131]:
%%writefile app/backend.py

import os
import re
import json
from pathlib import Path
from typing import Optional, List

from dotenv import load_dotenv
from pydantic import BaseModel

from langchain_groq import ChatGroq
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings


# =========================================================
# 1. PROJECT PATHS + API KEY
# =========================================================

ROOT_DIR = Path(__file__).resolve().parent.parent
CHROMA_DIR = ROOT_DIR / "chroma_db"

load_dotenv(ROOT_DIR / ".env")

if not os.getenv("GROQ_API_KEY"):
    raise ValueError(
        "GROQ_API_KEY not found. Check the .env file in the main project folder."
    )


# =========================================================
# 2. EMBEDDINGS + VECTOR DATABASE
# =========================================================

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma(
    persist_directory=str(CHROMA_DIR),
    embedding_function=embeddings,
    collection_name="biodiversity_knowledge"
)


# =========================================================
# 3. LLM
# =========================================================

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)


# =========================================================
# 4. DATA MODELS
# =========================================================

class EnvironmentalProfile(BaseModel):
    soil_organic_carbon: Optional[float] = None
    soil_ph: Optional[float] = None
    soil_moisture: Optional[str] = None

    rainfall: Optional[str] = None
    temperature: Optional[str] = None

    land_use: Optional[str] = None
    crop: Optional[str] = None

    biodiversity_trend: Optional[str] = None
    species_richness: Optional[str] = None
    habitat_diversity: Optional[str] = None

    pollution: Optional[str] = None
    deforestation: Optional[str] = None

    region: Optional[str] = None


class GroundingCheck(BaseModel):
    status: str
    supported_claims: List[str]
    unsupported_claims: List[str]
    explanation: str


extractor_llm = llm.with_structured_output(
    EnvironmentalProfile,
    method="json_schema"
)

grounding_llm = llm.with_structured_output(
    GroundingCheck,
    method="json_schema"
)


# =========================================================
# 5. EXTRACT ENVIRONMENTAL INFORMATION
# =========================================================

def extract_environmental_profile(user_text):

    prompt = f"""
Extract environmental information from the user's message.

USER MESSAGE:
{user_text}

Rules:
- Extract information explicitly stated by the user.
- Simple obvious category mappings are allowed.
- Do NOT invent measurements or environmental conditions.
- Unknown or missing information must be null.

Useful mappings:
- farm / wheat farm / crop field -> land_use = agriculture
- forest -> land_use = forest
- pasture / grazing land -> land_use = pasture
- semi arid -> region = semi-arid
- soil carbon is 0.3% -> soil_organic_carbon = 0.3
- biodiversity is declining -> biodiversity_trend = declining

Important:
- General biodiversity decline is NOT habitat diversity decline.
- General biodiversity decline is NOT species richness decline.
- Only populate habitat_diversity if explicitly discussed.
- Only populate species_richness if explicitly discussed.
"""

    return extractor_llm.invoke(prompt)


# =========================================================
# 6. MERGE CONVERSATION INFORMATION
# =========================================================

def merge_profiles(old_profile, new_profile):

    old_data = old_profile.model_dump()
    new_data = new_profile.model_dump()

    for key, value in new_data.items():
        if value is not None:
            old_data[key] = value

    return EnvironmentalProfile(**old_data)


# =========================================================
# 7. CHECK FOR MISSING INFORMATION
# =========================================================

IMPORTANT_FIELDS = {
    "soil_organic_carbon": "soil organic carbon percentage",
    "rainfall": "rainfall level or annual rainfall",
    "land_use": "current land use",
    "region": "region or climate type"
}


def find_missing_fields(profile):

    missing = []
    data = profile.model_dump()

    for field, description in IMPORTANT_FIELDS.items():
        if data.get(field) is None:
            missing.append((field, description))

    return missing


def generate_clarifying_question(profile):

    missing = find_missing_fields(profile)

    descriptions = [
        description
        for _, description in missing
    ]

    prompt = f"""
You are an environmental assessment assistant.

The following important information is missing:

{descriptions}

Ask the user concise and friendly follow-up questions.

Rules:
- Ask only for the missing information.
- Do not recommend interventions yet.
- Tell the user they can say "unknown" if they do not know.
- Keep the response short.
"""

    return llm.invoke(prompt).content


# =========================================================
# 8. FORMAT SCIENTIFIC EVIDENCE
# =========================================================

def format_evidence(docs):

    formatted = []

    for i, doc in enumerate(docs, 1):

        source = doc.metadata.get(
            "source_file",
            "Unknown"
        )

        page = doc.metadata.get(
            "page",
            "Unknown"
        )

        formatted.append(
            f"""
SOURCE {i}
File: {source}
Page: {page}

{doc.page_content}
"""
        )

    return "\n".join(formatted)


# =========================================================
# 9. REMOVE REFERENCE-HEAVY CHUNKS
# =========================================================

def is_reference_chunk(doc):

    text = doc.page_content.lower()

    et_al_count = text.count("et al.")

    year_count = len(
        re.findall(r"\b(19|20)\d{2}\b", text)
    )

    if "references" in text[:200]:
        return True

    if et_al_count >= 3 and year_count >= 4:
        return True

    return False


def retrieve_clean_evidence(
    query,
    k=15,
    final_k=6
):

    docs = vectorstore.similarity_search(
        query,
        k=k
    )

    clean_docs = [
        doc
        for doc in docs
        if not is_reference_chunk(doc)
    ]

    return clean_docs[:final_k]


# =========================================================
# 10. MULTI-METRIC ENVIRONMENTAL REASONING
# =========================================================

def analyze_environmental_profile(profile):

    profile_text = json.dumps(
        profile.model_dump(exclude_none=True),
        indent=2
    )

    query = f"""
Environmental biodiversity assessment.

Environmental profile:
{profile_text}

Retrieve scientific evidence about interactions between:
soil health,
soil organic carbon,
rainfall,
climate,
land use,
biodiversity,
habitat,
and human environmental pressures.
"""

    docs = retrieve_clean_evidence(
        query,
        k=15,
        final_k=8
    )

    evidence = format_evidence(docs)

    prompt = f"""
You are an evidence-grounded environmental scientist.

ENVIRONMENTAL PROFILE:
{profile_text}

SCIENTIFIC EVIDENCE:
{evidence}

Analyze MULTIPLE environmental variables together.

Do not simply discuss each variable independently.

Identify:
1. Main environmental risks
2. Important multi-metric interactions
3. Biodiversity implications
4. Reinforcing relationships or feedback loops

Rules:
- Scientific factual claims should use supplied evidence.
- Do not invent numerical effects.
- If a relationship is reasonable but not directly supported,
  clearly begin it with "Inference:".

Return:

### Environmental Diagnosis

### Multi-Metric Interactions

### Biodiversity Impact

### Evidence Used
"""

    response = llm.invoke(prompt)

    return response.content, docs


# =========================================================
# 11. RETRIEVE RECOMMENDATION EVIDENCE
# =========================================================

def build_recommendation_evidence(profile):

    agro_docs = retrieve_clean_evidence(
        """
agroforestry biodiversity conservation
climate regulation habitat connectivity agriculture
""",
        k=15,
        final_k=5
    )

    cover_docs = retrieve_clean_evidence(
        """
cover cropping crop rotation soil organic carbon
sustainable soil management agriculture
""",
        k=15,
        final_k=5
    )

    tillage_docs = retrieve_clean_evidence(
        """
conservation reduced tillage soil organic carbon
sustainable agriculture soil management
""",
        k=15,
        final_k=5
    )

    all_docs = (
        agro_docs
        + cover_docs
        + tillage_docs
    )

    unique_docs = []
    seen = set()

    for doc in all_docs:

        key = (
            doc.metadata.get("source_file"),
            doc.metadata.get("page"),
            doc.page_content[:100]
        )

        if key not in seen:
            seen.add(key)
            unique_docs.append(doc)

    return unique_docs


# =========================================================
# 12. GENERATE RECOMMENDATIONS
# =========================================================

def generate_submission_recommendations(
    profile,
    diagnosis
):

    profile_text = json.dumps(
        profile.model_dump(exclude_none=True),
        indent=2
    )

    docs = build_recommendation_evidence(profile)

    evidence = format_evidence(docs)

    prompt = f"""
You are a conservative scientific environmental
decision-support system.

ENVIRONMENTAL PROFILE:
{profile_text}

ENVIRONMENTAL DIAGNOSIS:
{diagnosis}

SCIENTIFIC EVIDENCE:
{evidence}

Generate exactly 3 practical recommendations.

STRICT RULES:

- State only what the retrieved evidence supports.
- Do not invent percentages.
- Do not invent species names.
- Do not invent field sizes.
- Do not invent numerical improvements.
- Do not invent exact scientific time-to-benefit claims.
- Never turn "listed as a management strategy"
  into "will definitely improve".
- Clearly label reasoning beyond direct evidence as "Inference:".

Prefer evidence-supported interventions such as:
- agroforestry
- cover cropping / crop rotation
- conservation or reduced tillage

For each recommendation provide:

### Recommendation

**Direct Evidence:**  
State what the retrieved documents directly support.

**Relevance to this Site:**  
Any reasoning beyond direct evidence must begin with:
"Inference:"

**Targeted Metrics:**  
Only include metrics supported by the retrieved evidence.

**Time Horizon:**  
Write:
"Planning estimate: Short / Medium / Long; not established by retrieved evidence."

**Confidence:**  
High / Medium / Low confidence in evidence support.

**Evidence:**  
List filename and page.

Finally provide:

### Priority Order

Clearly state:
"Decision-support judgement, not a scientifically proven universal ranking."
"""

    response = llm.invoke(prompt)

    return response.content, docs


# =========================================================
# 13. SCIENTIFIC GROUNDING CHECK
# =========================================================

def check_groundedness(answer, docs):

    evidence = format_evidence(docs)

    prompt = f"""
You are a scientific evidence verifier.

SCIENTIFIC EVIDENCE:
{evidence}

ANSWER:
{answer}

Evaluate whether the important scientific claims are supported
by the supplied evidence.

Rules:
- Paraphrasing is allowed.
- Tables, figures, headings and lists count as evidence.
- If a document lists cover cropping, crop rotations or
  conservation/reduced tillage as soil-carbon management
  strategies, reporting that is supported.
- Numerical claims require direct evidence.
- Specific species or exact quantitative outcomes require direct evidence.
- Clearly labelled "Inference:" statements are allowed as reasoning.
- Planning estimates do not make an answer ungrounded.
- Decision-support rankings do not make an answer ungrounded.

Choose exactly one status:

"Grounded"
= essentially all important scientific claims are supported.

"Partially Grounded"
= main recommendations are supported but some secondary
details are not directly supported.

"Not Grounded"
= main recommendations themselves lack evidence.

Return:
- status
- supported_claims
- unsupported_claims
- explanation
"""

    return grounding_llm.invoke(prompt)


# =========================================================
# 14. COMPLETE CHATBOT PIPELINE
# =========================================================

def process_message(
    user_message,
    current_profile=None
):

    if current_profile:
        profile = EnvironmentalProfile(
            **current_profile
        )
    else:
        profile = EnvironmentalProfile()

    # Extract new information
    new_profile = extract_environmental_profile(
        user_message
    )

    # Add it to remembered information
    profile = merge_profiles(
        profile,
        new_profile
    )

    # Check whether we need more information
    missing = find_missing_fields(profile)

    if missing:

        response = generate_clarifying_question(
            profile
        )

        return {
            "status": "needs_more_information",
            "profile": profile.model_dump(
                exclude_none=True
            ),
            "response": response,
            "grounding_status": None,
            "grounding_notes": []
        }

    # Environmental reasoning
    diagnosis, diagnosis_docs = (
        analyze_environmental_profile(profile)
    )

    # Evidence-backed recommendations
    recommendations, rec_docs = (
        generate_submission_recommendations(
            profile,
            diagnosis
        )
    )

    # Verification
    grounding = check_groundedness(
        recommendations,
        rec_docs
    )

    response = f"""
## Environmental Assessment

{diagnosis}

---

## Evidence-Backed Recommendations

{recommendations}
"""

    return {
        "status": "complete",

        "profile": profile.model_dump(
            exclude_none=True
        ),

        "response": response,

        "grounding_status": grounding.status,

        "grounding_notes": (
            grounding.unsupported_claims
        )
    }

Overwriting app/backend.py


In [132]:
%%writefile app/app.py

import streamlit as st
from backend import process_message


# =========================================================
# PAGE SETTINGS
# =========================================================

st.set_page_config(
    page_title="Biodiversity Intelligence",
    page_icon="🌿",
    layout="wide"
)


# =========================================================
# SESSION STATE
# =========================================================

if "messages" not in st.session_state:
    st.session_state.messages = []

if "profile" not in st.session_state:
    st.session_state.profile = {}

if "grounding_status" not in st.session_state:
    st.session_state.grounding_status = None

if "grounding_notes" not in st.session_state:
    st.session_state.grounding_notes = []


# =========================================================
# HEADER
# =========================================================

st.title("🌿 Biodiversity Intelligence")

st.caption(
    "AI-powered environmental decision support using "
    "scientific knowledge retrieval and multi-metric reasoning."
)


# =========================================================
# SIDEBAR
# =========================================================

with st.sidebar:

    st.header("Environmental Profile")

    if st.session_state.profile:

        for key, value in st.session_state.profile.items():

            label = key.replace("_", " ").title()

            st.write(f"**{label}:** {value}")

    else:

        st.info(
            "Environmental information will appear here "
            "as you describe your site."
        )

    st.divider()

    if st.session_state.grounding_status:

        st.subheader("Evidence Verification")

        status = st.session_state.grounding_status

        if status == "Grounded":
            st.success("Grounded")

        elif status == "Partially Grounded":
            st.warning("Partially Grounded")

        else:
            st.error("Not Grounded")

        if st.session_state.grounding_notes:

            with st.expander("Verification Notes"):

                for note in st.session_state.grounding_notes:
                    st.write("•", note)

    st.divider()

    if st.button("Reset Conversation"):

        st.session_state.messages = []
        st.session_state.profile = {}
        st.session_state.grounding_status = None
        st.session_state.grounding_notes = []

        st.rerun()


# =========================================================
# INTRO MESSAGE
# =========================================================

if not st.session_state.messages:

    st.info(
        """
Describe an ecosystem, farm, or land area.

Example:

**"Biodiversity is declining on my wheat farm."**

The assistant will ask for missing environmental information
and then generate an evidence-backed assessment.
"""
    )


# =========================================================
# DISPLAY CHAT HISTORY
# =========================================================

for message in st.session_state.messages:

    with st.chat_message(message["role"]):
        st.markdown(message["content"])


# =========================================================
# CHAT INPUT
# =========================================================

user_message = st.chat_input(
    "Describe your ecosystem or environmental concern..."
)

if user_message:

    # Save user message
    st.session_state.messages.append({
        "role": "user",
        "content": user_message
    })

    # Display user message
    with st.chat_message("user"):
        st.markdown(user_message)

    # Generate assistant response
    with st.chat_message("assistant"):

        with st.spinner(
            "Analyzing environmental evidence..."
        ):

            try:

                result = process_message(
                    user_message,
                    st.session_state.profile
                )

                response = result["response"]

                st.session_state.profile = (
                    result["profile"]
                )

                st.session_state.grounding_status = (
                    result["grounding_status"]
                )

                st.session_state.grounding_notes = (
                    result["grounding_notes"]
                )

                st.markdown(response)

            except Exception as e:

                response = (
                    "An error occurred while processing the request:\n\n"
                    f"`{e}`"
                )

                st.error(response)

    # Save assistant response
    st.session_state.messages.append({
        "role": "assistant",
        "content": response
    })

Writing app/app.py


In [133]:
!pip install -q streamlit python-dotenv

In [134]:
%%writefile requirements.txt
streamlit
python-dotenv
pydantic
langchain
langchain-community
langchain-text-splitters
langchain-chroma
langchain-huggingface
langchain-groq
chromadb
sentence-transformers
pypdf

Overwriting requirements.txt


In [135]:
%%writefile README.md
# 🌿 Biodiversity Intelligence

AI-powered environmental decision-support chatbot built for the **Darukaa.Earth AI Biodiversity Intelligence Chatbot Hackathon**.

The system combines scientific retrieval, multi-metric environmental reasoning, conversation memory, evidence-backed recommendations, and scientific grounding verification.

---

## Project Objective

Environmental problems depend on multiple connected variables such as soil health, rainfall, land use, climate, biodiversity, and human environmental pressures.

This project is designed to behave more like an AI environmental scientist than a generic chatbot.

The system can:

- Understand environmental conditions from natural-language input
- Build a structured environmental profile
- Ask follow-up questions when important information is missing
- Remember information across multiple conversation turns
- Retrieve relevant scientific evidence
- Perform multi-metric environmental reasoning
- Generate evidence-backed recommendations
- Cite scientific source documents and page numbers
- Verify scientific claims against retrieved evidence

---

## System Workflow

User Input  
↓  
Environmental Information Extraction  
↓  
Structured Environmental Profile  
↓  
Conversation Memory  
↓  
Missing Information Check  
↓  
Clarifying Questions if Needed  
↓  
Scientific RAG Retrieval  
↓  
Chroma Vector Database  
↓  
Relevant Scientific Evidence  
↓  
Multi-Metric Environmental Reasoning  
↓  
Evidence-Backed Recommendations  
↓  
Scientific Grounding Verification  
↓  
Final Environmental Decision Support

---

## Environmental Variables Supported

The environmental profile supports:

- Soil organic carbon
- Soil pH
- Soil moisture
- Rainfall
- Temperature
- Land use
- Crop type
- Biodiversity trend
- Species richness
- Habitat diversity
- Pollution
- Deforestation
- Region / climate type

---

## Multi-Turn Conversation Example

### Message 1

Biodiversity is declining on my wheat farm.

The system extracts:

- Land use: Agriculture
- Crop: Wheat
- Biodiversity trend: Declining

### Message 2

Soil organic carbon is 0.3% and rainfall is low.

The system adds:

- Soil organic carbon: 0.3%
- Rainfall: Low

### Message 3

The farm is in a semi-arid region.

The completed profile contains information such as:

- Soil organic carbon: 0.3%
- Rainfall: Low
- Land use: Agriculture
- Crop: Wheat
- Biodiversity trend: Declining
- Region: Semi-arid

Once enough information is available, the scientific assessment begins.

---

## Scientific Knowledge Base

The project uses a **Retrieval-Augmented Generation (RAG)** architecture.

Scientific PDF documents are stored inside:

`knowledge_base/`

The knowledge base contains scientific environmental material related to:

- Soil organic carbon
- Soil health
- Soil biodiversity
- Agroforestry
- Climate change
- Biodiversity
- Sustainable agriculture
- Land management
- Ecosystem restoration

The exact PDF filenames may vary.

The application uses metadata stored with the vector database so retrieved evidence can still display the actual source filename and page number.

---

## RAG Pipeline

Scientific PDF Documents  
↓  
PDF Text Extraction  
↓  
Text Chunking  
↓  
Sentence Transformer Embeddings  
↓  
Chroma Vector Database  
↓  
Semantic Similarity Search  
↓  
Relevant Scientific Evidence  
↓  
Environmental Reasoning

### Embedding Model

`sentence-transformers/all-MiniLM-L6-v2`

### Vector Database

`ChromaDB`

The vector database is stored locally inside:

`chroma_db/`

---

## Evidence Filtering

Scientific documents can contain bibliography-heavy or reference-heavy sections.

The retrieval pipeline filters reference-heavy chunks before passing evidence to the reasoning model.

This helps improve:

- Retrieval quality
- Scientific relevance
- Token efficiency
- Grounding quality

---

## Multi-Metric Environmental Reasoning

The chatbot reasons across multiple environmental variables together rather than treating each variable independently.

For example, it may analyze interactions between:

- Soil organic carbon
- Soil biodiversity
- Rainfall
- Agricultural land use
- Crop type
- Climate
- Habitat
- Biodiversity trend

When reasoning goes beyond direct scientific evidence, the system labels it as:

`Inference:`

---

## Evidence-Backed Recommendations

The current system generates three environmental intervention categories:

1. Agroforestry
2. Cover Cropping / Crop Rotation
3. Conservation / Reduced Tillage

Each recommendation includes:

- Direct Evidence
- Relevance to the Site
- Targeted Metrics
- Time Horizon
- Confidence
- Scientific source filename and page
- Priority Order

---

## Scientific Grounding Verification

The system does not automatically trust its own generated recommendations.

A separate verification stage compares generated scientific claims with the retrieved evidence.

Possible statuses include:

- Grounded
- Partially Grounded
- Not Grounded
- Verification Failed

This helps reduce hallucination and makes the environmental reasoning more transparent.

---

## Reliability Features

The application includes:

- Scientific RAG retrieval
- Source and page-level references
- Reference-heavy chunk filtering
- Deterministic environmental field extraction
- Multi-turn profile memory
- Missing-information detection
- Explicit inference labelling
- Conservative recommendation prompts
- Separate scientific grounding verification
- Incomplete recommendation detection
- Automatic retry handling for temporary Groq rate limits
- Preservation of user-provided units

---

## Language Model

The application uses:

`openai/gpt-oss-20b`

through the Groq API.

The LLM is mainly used for:

- Multi-metric environmental reasoning
- Recommendation generation
- Scientific grounding verification

Simple environmental information extraction is handled locally where possible to improve reliability and reduce token usage.

---

## User Interface

The application uses **Streamlit**.

The interface includes:

- Conversational chat
- Environmental profile sidebar
- Clarifying questions
- Environmental assessment
- Evidence-backed recommendations
- Scientific grounding status
- Verification notes
- Conversation reset functionality

---

## Project Structure

Biodiversity Chatbot/

- app/
  - app.py
  - backend.py
- knowledge_base/
  - scientific PDF documents
- chroma_db/
- darukaa_biodiversity.ipynb
- README.md
- requirements.txt
- .gitignore
- .env

The exact scientific PDF filenames inside `knowledge_base/` may differ depending on the source documents used during indexing.

---

## Installation

Install the required Python packages:

`pip install -r requirements.txt`

---

## Environment Variable

Create a `.env` file in the main project folder containing:

`GROQ_API_KEY=your_groq_api_key_here`

Do not upload the `.env` file to GitHub.

---

## Run the Application

Run:

`streamlit run app/app.py`

For the Anaconda environment used during development on Windows:

`& "C:\Users\krish\anaconda3\python.exe" -m streamlit run app/app.py`

---

## Recommended .gitignore

The `.gitignore` file should contain:

.env  
__pycache__/  
*.pyc  
.ipynb_checkpoints/  
.DS_Store

---

## Main Demo Scenario

Use the following messages one by one:

1. Biodiversity is declining on my wheat farm.
2. Soil organic carbon is 0.3% and rainfall is low.
3. The farm is in a semi-arid region.

This demonstrates:

- Natural-language understanding
- Structured environmental profile extraction
- Clarifying questions
- Multi-turn memory
- Multi-variable environmental reasoning
- Scientific RAG retrieval
- Evidence-backed recommendations
- Source citations
- Grounding verification

---

## Limitations

- The chatbot is a decision-support tool and is not a replacement for field measurements or professional environmental assessment.
- Environmental outcomes can vary by geography and local conditions.
- Planning time horizons are estimates unless explicitly supported by retrieved evidence.
- Scientific grounding depends on the quality and coverage of the documents in the knowledge base.
- Geographic coordinate analysis is not currently implemented.

---

## Technology Stack

- Python
- Streamlit
- LangChain
- ChromaDB
- HuggingFace Sentence Transformers
- Groq API
- Pydantic
- PyPDF
- Retrieval-Augmented Generation
- Scientific grounding verification

---

## Project Goal

The goal is to provide:

**Retrievable, explainable, evidence-backed environmental intelligence.**

---

Developed for the **Darukaa.Earth AI/ML Engineer Hackathon**.

Writing README.md


In [136]:
%%writefile .gitignore
.env
__pycache__/
*.pyc
.ipynb_checkpoints/
.DS_Store

Overwriting .gitignore


In [137]:
import os

for item in os.listdir("."):
    print(item)

.env
.gitignore
.ipynb_checkpoints
app
chroma_db
darukaa_biodiversity.ipynb
data
knowledge_base
README.md
requirements.txt
